In [13]:
import geopandas as gpd
import pandas as pd

# Path to the main OSM snapshot of vet clinics in Berlin
# Note: the notebook is already in the `veterinary_clinics` directory,
# so we only need to point to "sources/...".
osm_geojson_path = "sources/raw_osm_berlin_vet_clinics_20251209.geojson"

# Load vet clinics from OpenStreetMap snapshot
gdf_vets = gpd.read_file(osm_geojson_path)

# Quick sanity check: first rows and basic shape
display(gdf_vets.head())
print(f"Number of OSM vet clinic features: {len(gdf_vets)}")

,id,@id,addr:city,addr:country,addr:district,addr:floor,addr:housenumber,addr:place,addr:postcode,addr:street,...,veterinary:treats:dog,veterinary:treats:reptile,veterinary:treats:small_mammals,website,wheelchair,wheelchair:description,wikidata,wikimedia_commons,@geometry,geometry
0,way/24921047,way/24921047,Berlin,DE,None,None,136,None,13405,Scharnweberstraße,...,None,None,None,https://www.vetzentrum-berlin.de/,None,None,None,None,center,POINT (13.3259 52.56384)
1,way/28608972,way/28608972,Berlin,DE,None,None,37,None,13407,Alt-Reinickendorf,...,None,None,None,None,no,None,Q15107420,Category:Bauernhof Großkopf,center,POINT (13.35128 52.57485)
2,way/71173694,way/71173694,Berlin,DE,None,None,258,None,12679,Märkische Allee,...,None,None,None,https://www.tierklinik-in-berlin.de/,yes,None,None,None,center,POINT (13.55325 52.55556)
3,way/89101208,way/89101208,Berlin,DE,None,None,90,None,10318,Robert-Siewert-Straße,...,None,None,None,None,limited,None,None,None,center,POINT (13.53692 52.49265)
4,way/117230559,way/117230559,Berlin,DE,None,None,21,None,14193,Winkler Straße,...,None,None,None,None,None,None,Q76599062,None,center,POINT (13.26661 52.48762)


Number of OSM vet clinic features: 175


In [14]:
# Path to Berlin LOR / Ortsteile polygons
lor_path = "sources/raw_berlin_lor_ortsteile.geojson"

# Load LOR polygons
gdf_lor = gpd.read_file(lor_path)

display(gdf_lor.head())
print("LOR columns:", list(gdf_lor.columns))

,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA,geometry
0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,0102,Moabit,Polygon,Moabit,Mitte,768.7909,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,0103,Hansaviertel,Polygon,Hansaviertel,Mitte,52.5337,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,0104,Tiergarten,Polygon,Tiergarten,Mitte,516.0672,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,0105,Wedding,Polygon,Wedding,Mitte,919.9112,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


LOR columns: ['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL', 'BEZIRK', 'FLAECHE_HA', 'geometry']


In [15]:
gdf_lor.columns

Index(['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL',
       'BEZIRK', 'FLAECHE_HA', 'geometry'],
      dtype='object')

In [16]:
# Check CRS (both should be WGS84 / EPSG:4326)
print("Vet clinics CRS:", gdf_vets.crs)
print("LOR CRS:", gdf_lor.crs)

Vet clinics CRS: EPSG:4326
LOR CRS: EPSG:4326


In [17]:
# Check CRS (both should be WGS84 / EPSG:4326)
print("Vet clinics CRS:", gdf_vets.crs)
print("LOR CRS:", gdf_lor.crs)

Vet clinics CRS: EPSG:4326
LOR CRS: EPSG:4326


In [18]:
# Column names in the LOR dataset
lor_id_col = "gml_id"          # LOR ID
district_col = "BEZIRK"        # district name
neighborhood_col = "OTEIL"     # neighborhood / Ortsteil name

cols_lor_keep = [lor_id_col, district_col, neighborhood_col, "geometry"]

# Subset of LOR GeoDataFrame with only the relevant attributes
gdf_lor_subset = gdf_lor[cols_lor_keep].copy()

display(gdf_lor_subset.head())

,gml_id,BEZIRK,OTEIL,geometry
0,re_ortsteil.0101,Mitte,Mitte,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,Mitte,Moabit,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,Mitte,Hansaviertel,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,Mitte,Tiergarten,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,Mitte,Wedding,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


In [19]:
# Spatial join: vet clinics (points) WITHIN LOR polygons
gdf_vets_lor = gpd.sjoin(
    gdf_vets,
    gdf_lor_subset,
    how="left",
    predicate="within",
)

display(gdf_vets_lor.head())
print(f"Joined GeoDataFrame shape: {gdf_vets_lor.shape}")

,id,@id,addr:city,addr:country,addr:district,addr:floor,addr:housenumber,addr:place,addr:postcode,addr:street,...,wheelchair,wheelchair:description,wikidata,wikimedia_commons,@geometry,geometry,index_right,gml_id,BEZIRK,OTEIL
0,way/24921047,way/24921047,Berlin,DE,None,None,136,None,13405,Scharnweberstraße,...,None,None,None,None,center,POINT (13.3259 52.56384),85,re_ortsteil.1201,Reinickendorf,Reinickendorf
1,way/28608972,way/28608972,Berlin,DE,None,None,37,None,13407,Alt-Reinickendorf,...,no,None,Q15107420,Category:Bauernhof Großkopf,center,POINT (13.35128 52.57485),85,re_ortsteil.1201,Reinickendorf,Reinickendorf
2,way/71173694,way/71173694,Berlin,DE,None,None,258,None,12679,Märkische Allee,...,yes,None,None,None,center,POINT (13.55325 52.55556),70,re_ortsteil.1001,Marzahn-Hellersdorf,Marzahn
3,way/89101208,way/89101208,Berlin,DE,None,None,90,None,10318,Robert-Siewert-Straße,...,limited,None,None,None,center,POINT (13.53692 52.49265),76,re_ortsteil.1102,Lichtenberg,Karlshorst
4,way/117230559,way/117230559,Berlin,DE,None,None,21,None,14193,Winkler Straße,...,None,None,Q76599062,None,center,POINT (13.26661 52.48762),24,re_ortsteil.0404,Charlottenburg-Wilmersdorf,Grunewald


Joined GeoDataFrame shape: (175, 84)


In [20]:
# Rename LOR columns to clean English names
rename_map = {
    "gml_id": "lor_id",
    "BEZIRK": "district_name",
    "OTEIL": "neighborhood_name",
}

gdf_vets_lor = gdf_vets_lor.rename(columns=rename_map)

# Add explicit longitude and latitude columns from geometry
gdf_vets_lor["lon"] = gdf_vets_lor.geometry.x
gdf_vets_lor["lat"] = gdf_vets_lor.geometry.y

# Quick check of the enriched data
display(
    gdf_vets_lor[["name", "lat", "lon", "district_name", "neighborhood_name"]]
    .head()
)

,name,lat,lon,district_name,neighborhood_name
0,Das Veterinärmedizinische Zentrum Berlin,52.563836,13.325898,Reinickendorf,Reinickendorf
1,Zete Marton,52.574848,13.351279,Reinickendorf,Reinickendorf
2,Tierärztlichen Klinik für Kleintiere,52.555555,13.553248,Marzahn-Hellersdorf,Marzahn
3,Tierarztpraxis Kathrin Böhm,52.492651,13.536917,Lichtenberg,Karlshorst
4,Tierartzpraxis Gotthardt,52.487622,13.266606,Charlottenburg-Wilmersdorf,Grunewald


In [21]:
# 6.1 Select columns to export for the v0 dataset

cols_export = [
    "id",
    "@id",
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    # contact details (only included if present in the OSM export)
    "phone" if "phone" in gdf_vets_lor.columns else None,
    "contact:phone" if "contact:phone" in gdf_vets_lor.columns else None,
    "website" if "website" in gdf_vets_lor.columns else None,
    "contact:website" if "contact:website" in gdf_vets_lor.columns else None,
    "email" if "email" in gdf_vets_lor.columns else None,
    "contact:email" if "contact:email" in gdf_vets_lor.columns else None,
    # opening hours and operator / brand info
    "opening_hours" if "opening_hours" in gdf_vets_lor.columns else None,
    "operator" if "operator" in gdf_vets_lor.columns else None,
    "brand" if "brand" in gdf_vets_lor.columns else None,
    # spatial context
    "lor_id",
    "district_name",
    "neighborhood_name",
    # geometry-derived coordinates
    "lat",
    "lon",
]

# Remove None entries and any columns that do not exist (defensive programming)
cols_export = [c for c in cols_export if c is not None and c in gdf_vets_lor.columns]

print("Columns to be exported:")
print(cols_export)
print(f"Total columns: {len(cols_export)}")

# 6.2 Export to CSV (v0)
output_path = "cache/vets_osm_berlin_with_lor_20251209_v0.csv"

gdf_vets_lor[cols_export].to_csv(output_path, index=False)

print(f"\nExported v0 vet clinics dataset to: {output_path}")

# Quick sample of the exported data
gdf_vets_lor[cols_export].head()

Columns to be exported:
['id', '@id', 'name', 'addr:street', 'addr:housenumber', 'addr:postcode', 'addr:city', 'phone', 'contact:phone', 'website', 'contact:website', 'email', 'contact:email', 'opening_hours', 'operator', 'lor_id', 'district_name', 'neighborhood_name', 'lat', 'lon']
Total columns: 20

Exported v0 vet clinics dataset to: cache/vets_osm_berlin_with_lor_20251209_v0.csv


,id,@id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,website,contact:website,email,contact:email,opening_hours,operator,lor_id,district_name,neighborhood_name,lat,lon
0,way/24921047,way/24921047,Das Veterinärmedizinische Zentrum Berlin,Scharnweberstraße,136,13405,Berlin,+49 30 4127357,None,https://www.vetzentrum-berlin.de/,None,info@vetzentrum-berlin.de,None,24/7,Kai S. Rödiger,re_ortsteil.1201,Reinickendorf,Reinickendorf,52.563836,13.325898
1,way/28608972,way/28608972,Zete Marton,Alt-Reinickendorf,37,13407,Berlin,None,None,None,None,None,None,"Mo-Fr 09:00-12:00, Mo,Th 16:00-19:00, Tu,We,Fr...",None,re_ortsteil.1201,Reinickendorf,Reinickendorf,52.574848,13.351279
2,way/71173694,way/71173694,Tierärztlichen Klinik für Kleintiere,Märkische Allee,258,12679,Berlin,None,None,https://www.tierklinik-in-berlin.de/,None,None,None,24/7,None,re_ortsteil.1001,Marzahn-Hellersdorf,Marzahn,52.555555,13.553248
3,way/89101208,way/89101208,Tierarztpraxis Kathrin Böhm,Robert-Siewert-Straße,90,10318,Berlin,+49 30 32669736,None,None,None,None,None,"Th 10:00-18:00; Tu,We 10:00-19:00; Fr 10:00-15...",Kathrin Böhm,re_ortsteil.1102,Lichtenberg,Karlshorst,52.492651,13.536917
4,way/117230559,way/117230559,Tierartzpraxis Gotthardt,Winkler Straße,21,14193,Berlin,None,None,None,None,None,None,None,None,re_ortsteil.0404,Charlottenburg-Wilmersdorf,Grunewald,52.487622,13.266606
